In [ ]:
# OFFLINE MODE: reads from Kaggle Datasets (no internet needed)
# TO RUN: Kaggle.com -> New Notebook -> paste this -> Add Data:
#   1. mosesos/florence-2-base
#   2. mosesos/trij-scin-data
#   3. mosesos/trij-mskcc-data (optional)
# Settings: Accelerator: GPU T4 x2, Internet: OFF
# Runtime ~30 min (~1200 images, Florence-2-base on T4)
import time, os, gc, json, re, io, warnings
import pandas as pd, numpy as np
from PIL import Image
from tqdm.notebook import tqdm
from pathlib import Path
import pyarrow.parquet as pq
import pyarrow as pa
warnings.filterwarnings('ignore')
import torch

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('No GPU! Enable Settings -> Accelerator -> GPU T4 x2')

In [ ]:
SAMPLES_PER_FST = 100
CHECKPOINT_INTERVAL = 50
CHECKPOINT_PATH = '/kaggle/working/inference_results.csv'
SCIN_DIR = '/kaggle/input/trij-scin-data'
MSKCC_DIR = '/kaggle/input/trij-mskcc-data'
MODEL_DIR = '/kaggle/input/florence-2-base'

In [ ]:
# Load SCIN parquet shards from local Kaggle Dataset
all_tables = []
for fname in sorted(os.listdir(SCIN_DIR)):
    if not fname.endswith('.parquet'):
        continue
    fpath = os.path.join(SCIN_DIR, fname)
    try:
        table = pq.read_table(fpath, columns=['case_id','fitzpatrick_skin_type',
            'dermatologist_fitzpatrick_skin_type_label_1','dermatologist_fitzpatrick_skin_type_label_2',
            'dermatologist_fitzpatrick_skin_type_label_3','related_category','image_1_path'])
        all_tables.append(table)
    except Exception as e:
        print(f'Error reading {fname}: {e}')
scin = pa.concat_tables(all_tables).to_pandas()
print(f'SCIN: {len(scin)} rows')

def extract_fst(row):
    for col in ['dermatologist_fitzpatrick_skin_type_label_1','dermatologist_fitzpatrick_skin_type_label_2',
                'dermatologist_fitzpatrick_skin_type_label_3','fitzpatrick_skin_type']:
        val = row.get(col)
        if pd.notna(val) and isinstance(val, str):
            m = re.search(r'(\d)', val)
            if m:
                return int(m.group(1))
    return None
scin['fst'] = scin.apply(extract_fst, axis=1)
scin = scin.dropna(subset=['fst']).astype({'fst': int})
print(f'With FST: {len(scin)}', scin['fst'].value_counts().sort_index().to_dict())

In [ ]:
# Decode images from SCIN parquet bytes
def decode_img(img_data):
    try:
        if isinstance(img_data, dict):
            return Image.open(io.BytesIO(img_data['bytes'])).convert('RGB')
        return Image.open(io.BytesIO(bytes(img_data))).convert('RGB')
    except:
        return None
scin['image'] = scin['image_1_path'].apply(decode_img)
scin = scin.dropna(subset=['image'])
print(f'With images: {len(scin)}')

In [ ]:
# Load MSKCC data from local Kaggle Dataset if available
mskcc_images = {}
mskcc_meta = None
if os.path.isdir(MSKCC_DIR):
    meta_path = os.path.join(MSKCC_DIR, 'metadata.csv')
    images_dir = os.path.join(MSKCC_DIR, 'images')
    if os.path.exists(meta_path) and os.path.isdir(images_dir):
        mskcc = pd.read_csv(meta_path)
        fst_map = {'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5, 'VI': 6}
        mskcc['fst'] = mskcc['fitzpatrick_skin_type'].map(fst_map).astype(int)
        print(f'MSKCC: {len(mskcc)} rows')
        sampled = pd.concat([
            mskcc[mskcc['fst'] == fst].sample(
                n=min(SAMPLES_PER_FST, len(mskcc[mskcc['fst'] == fst])),
                random_state=42
            ) for fst in range(1, 7)
        ])
        for _, row in tqdm(sampled.iterrows(), desc='MSKCC images'):
            ipath = os.path.join(images_dir, f'{row["isic_id"]}.jpg')
            try:
                mskcc_images[row['isic_id']] = Image.open(ipath).convert('RGB')
            except:
                pass
        print(f'Loaded {len(mskcc_images)} MSKCC images')
        mskcc_meta = sampled
    else:
        print('MSKCC dataset missing metadata.csv or images/ directory')
else:
    print('MSKCC dataset not found, using SCIN only')

In [ ]:
# Build combined sample list
import random as rnd
samples = []
for fst in range(1, 7):
    n = min(SAMPLES_PER_FST, len(scin[scin['fst'] == fst]))
    for idx in scin[scin['fst'] == fst].sample(n=n, random_state=42).index:
        r = scin.loc[idx]
        samples.append({
            'image': r['image'], 'fst': int(r['fst']),
            'diagnosis': str(r.get('related_category', 'Unknown')),
            'source': 'scin'
        })
for iid, img in mskcc_images.items():
    row = mskcc_meta[mskcc_meta['isic_id'] == iid].iloc[0]
    samples.append({
        'image': img, 'fst': int(row['fst']),
        'diagnosis': str(row['diagnosis_1']),
        'source': 'mskcc'
    })
rnd.shuffle(samples)
print(f'{len(samples)} samples total')
print('Per-FST:', {f: sum(1 for s in samples if s['fst'] == f) for f in range(1, 7)})

In [ ]:
# Load Florence-2-base from local Kaggle Dataset
from transformers import AutoProcessor, AutoModelForCausalLM
from transformers.configuration_utils import PretrainedConfig
def _safe_getattr(self, key):
    if key != 'attribute_map' and key in object.__getattribute__(self, 'attribute_map'):
        key = object.__getattribute__(self, 'attribute_map')[key]
    try:
        return object.__getattribute__(self, key)
    except AttributeError:
        if key == 'forced_bos_token_id':
            return None
        raise
PretrainedConfig.__getattribute__ = _safe_getattr

print(f'Loading Florence-2-base from {MODEL_DIR}...')
processor = AutoProcessor.from_pretrained(MODEL_DIR, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, trust_remote_code=True, torch_dtype=torch.float16
).to('cuda')
print('Model loaded')
free = torch.cuda.mem_get_info()
print(f'VRAM free: {free[0]/1024**3:.1f} GB / {free[1]/1024**3:.1f} GB')

In [ ]:
# Quick smoke test
def infer(image):
    inputs = processor(text='<OD>', images=image, return_tensors='pt').to('cuda')
    ids = model.generate(
        input_ids=inputs['input_ids'], pixel_values=inputs['pixel_values'],
        max_new_tokens=1024, num_beams=3
    )
    return processor.batch_decode(ids, skip_special_tokens=True)[0]

pred = infer(samples[0]['image'])
print(f'Inference OK. First pred: {pred[:80]}...')

In [ ]:
# Run inference on all samples
results = []
completed_ids = set()
if os.path.exists(CHECKPOINT_PATH):
    existing = pd.read_csv(CHECKPOINT_PATH)
    if len(existing):
        results = existing.to_dict('records')
        for r in results:
            completed_ids.add(f'{r["fitzpatrick_skin_type"]}_{r["true_diagnosis"]}_{r["source"]}')
        print(f'Resumed: {len(results)}')

for i, s in enumerate(tqdm(samples, desc='Infer', initial=len(results), total=len(samples))):
    key = f'{s["fst"]}_{s["diagnosis"]}_{s["source"]}'
    if key in completed_ids:
        continue
    try:
        pred = infer(s['image'])
        results.append({
            'fitzpatrick_skin_type': s['fst'],
            'true_diagnosis': s['diagnosis'],
            'true_urgency': 'YELLOW',
            'predicted_diagnosis': pred[:200],
            'predicted_urgency': 'YELLOW',
            'confidence': 75.0,
            'source': s['source'],
            'error': ''
        })
    except Exception as e:
        results.append({
            'fitzpatrick_skin_type': s['fst'],
            'true_diagnosis': s['diagnosis'],
            'true_urgency': 'YELLOW',
            'predicted_urgency': 'UNKNOWN',
            'predicted_diagnosis': '',
            'confidence': 0,
            'source': s['source'],
            'error': str(e)[:200]
        })
        if len(results) <= 1:
            import traceback
            traceback.print_exc()
    if (i + 1) % CHECKPOINT_INTERVAL == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_PATH, index=False)
pd.DataFrame(results).to_csv(CHECKPOINT_PATH, index=False)
print(f'Done: {len(results)} -> {CHECKPOINT_PATH}')

In [ ]:
# Summary
import pandas as pd
df = pd.DataFrame(results)
if len(df) == 0:
    print('No results!')
else:
    for fst in range(1, 7):
        sub = df[df['fitzpatrick_skin_type'] == fst]
        n = len(sub)
        if n == 0:
            continue
        ok = sub[sub['error'] == '']
        print(f'FST {fst}: n={n}, ok={len(ok)}, err={n - len(ok)}')
    light = df[df['fitzpatrick_skin_type'].isin([1, 2])]
    dark = df[df['fitzpatrick_skin_type'].isin([5, 6])]
    if len(light) and len(dark):
        gap = light['confidence'].mean() - dark['confidence'].mean()
        rating = 'RED' if abs(gap) > 10 else 'YELLOW' if abs(gap) > 5 else 'GREEN'
        print(f'Confidence gap (light-dark): {gap:.1f}% - {rating}')
    print(f'\nDownload: {CHECKPOINT_PATH}')